In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install openimages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 117.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.9 MB/s eta 0:00:00


In [3]:
DRIVE_ROOT = "/content"
DATA_SRC = DRIVE_ROOT + "/masters/gmm/lab2/data"
RAW_DATA_SRC = DATA_SRC + "/raw"

In [4]:
!oi_download_images --base_dir $RAW_DATA_SRC --labels Car --limit 500

2026-03-16  20:41:24 INFO Downloading 500 train images for class 'car'
100% 500/500 [00:13<00:00, 36.34it/s]


In [5]:
!oi_download_images --base_dir $RAW_DATA_SRC --labels Cat --limit 500

2026-03-16  20:42:06 INFO Downloading 500 train images for class 'cat'
100% 500/500 [00:14<00:00, 35.13it/s]


In [6]:
!oi_download_images --base_dir $RAW_DATA_SRC --labels Dog --limit 500

2026-03-16  20:42:48 INFO Downloading 500 train images for class 'dog'
 98% 491/500 [00:13<00:00, 16.98it/s]2026-03-16  20:43:02 WARNING Connection pool is full, discarding connection: open-images-dataset.s3.amazonaws.com. Connection pool size: 10
100% 500/500 [00:13<00:00, 35.89it/s]


In [7]:
!ls $RAW_DATA_SRC/car/images | wc -l

500


In [8]:
!ls $RAW_DATA_SRC/cat/images | wc -l

500


In [9]:
!ls $RAW_DATA_SRC/dog/images | wc -l

500


In [10]:
import os
import shutil
import random

RAW_DIR = os.path.join(DATA_SRC, "raw") 

TRAIN_DIR = os.path.join(DATA_SRC, "train")
VAL_DIR = os.path.join(DATA_SRC, "val")
TEST_DIR = os.path.join(DATA_SRC, "test")

# Classes (lowercase)
CLASSES = ["cat", "dog", "car"]

# Split ratios
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Ensure train/val/test folders exist
for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(split_dir, exist_ok=True)
    for cls in CLASSES:
        os.makedirs(os.path.join(split_dir, cls), exist_ok=True)

# Function to move and split images
def split_class_images(class_name):
    src_dir = os.path.join(RAW_DIR, class_name, 'images')
    if not os.path.exists(src_dir):
        print(f"Warning: {src_dir} does not exist. Skipping {class_name}.")
        return

    images = [f for f in os.listdir(src_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not images:
        print(f"No images found for {class_name} in {src_dir}")
        return

    random.shuffle(images)
    n = len(images)
    train_end = int(TRAIN_SPLIT * n)
    val_end = int((TRAIN_SPLIT + VAL_SPLIT) * n)

    splits = {
        TRAIN_DIR: images[:train_end],
        VAL_DIR: images[train_end:val_end],
        TEST_DIR: images[val_end:]
    }

    for split_dir, split_imgs in splits.items():
        target_dir = os.path.join(split_dir, class_name)
        os.makedirs(target_dir, exist_ok=True)
        for img in split_imgs:
            shutil.move(os.path.join(src_dir, img), os.path.join(target_dir, img))

for cls in CLASSES:
    print(f"Processing class: {cls}")
    split_class_images(cls)
    
def count_images(folder):
    for cls in CLASSES:
        cls_dir = os.path.join(folder, cls)
        if os.path.exists(cls_dir):
            print(f"{cls}: {len(os.listdir(cls_dir))}")
        else:
            print(f"{cls}: 0")

print("\nTrain dataset counts:")
count_images(TRAIN_DIR)
print("\nValidation dataset counts:")
count_images(VAL_DIR)
print("\nTest dataset counts:")
count_images(TEST_DIR)

Processing class: cat
Processing class: dog
Processing class: car

Train dataset counts:
cat: 350
dog: 350
car: 350

Validation dataset counts:
cat: 75
dog: 75
car: 75

Test dataset counts:
cat: 75
dog: 75
car: 75
